# Workflow

Build a parcel handoff coordinator for a regional delivery team. When a temperature-sensitive parcel has had no scan for six hours, open a manual trace before dispatch and mark the handoff record for review.

In [ ]:
from typing import Any, TypedDict

from langgraph.graph import END, START, StateGraph

In [ ]:
CONFIG = {"configurable": {"thread_id": "fixture-session"}, "recursion_limit": 10}

In [ ]:
class WorkflowState(TypedDict, total=False):
    request: dict[str, Any]
    result: Any

In [ ]:
def prepare_handoff(tracking_id: str, temperature_sensitive: bool, last_scan_hours: int) -> dict[str, object]:
    record = {
        'tracking_id': tracking_id,
        'temperature_sensitive': temperature_sensitive,
        'last_scan_hours': last_scan_hours,
    }
    if temperature_sensitive and last_scan_hours >= 6:
        record['manual_trace_required'] = True
        record['dispatch_review'] = 'human_check'
    return record

def perform_request(state: WorkflowState) -> WorkflowState:
    return {"result": prepare_handoff(**state["request"])}

In [ ]:
builder = StateGraph(WorkflowState)
builder.add_node("perform_request", perform_request)
builder.add_edge(START, "perform_request")
builder.add_edge("perform_request", END)
graph = builder.compile()

In [ ]:
def run_workflow(request: dict[str, Any]) -> Any:
    result = graph.invoke({"request": request}, config=CONFIG)
    return result["result"]

The workflow returns the result from the requested operation.

Supply the operation's named arguments in the request dictionary.